# PyTorch — Build Your Own GPT

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kader-xai/ml-course-notebooks/blob/main/pytorch_course.ipynb)

Code from all 19 episodes of the PyTorch video series, from a single tensor up to a tiny GPT.

> Episodes PT01–PT14 run standalone. The GPT build (PT15–PT19) is shown **modular** exactly as in the videos (`model.py`, `block.py`, `train.py`, `generate.py`); the final cell consolidates them into one **runnable** tiny char-GPT you can train and sample in Colab.

In [ ]:
!pip -q install datasets  # torch is preinstalled on Colab

## PT01 · Why PyTorch (and What We're Building)

In [ ]:
import torch

print(f"PyTorch {torch.__version__}", end=" · ")
print(f"CUDA: {torch.cuda.is_available()}  MPS: {torch.backends.mps.is_available()}")

# build a 2x3 float tensor
x = torch.tensor([[1., 2., 3.],
                  [4., 5., 6.]])

# pick the best device: CUDA > MPS > CPU
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
x = x.to(device)

# one elementwise op, one matmul
doubled = x * 2
w = torch.tensor([[1., 0.], [0., 1.], [1., 1.]], device=device)
out = x @ w

print(doubled)
print(out)
print(f"shape={out.shape} · dtype={out.dtype} · device={out.device}")
print("Building toward: a model that finishes your sentences.")

## PT02 · Tensors — The Only Data Type That Matters

In [ ]:
"""tensors.py — create, type, reshape, and index any tensor."""
import torch

# ── 1 · Create tensors ──────────────────────────────
x = torch.tensor([[1.0, 2.0, 3.0],
                  [4.0, 5.0, 6.0]])      # from a Python list
z = torch.zeros(2, 3)                    # all zeros
o = torch.ones(2, 3)                     # all ones
r = torch.arange(6)                      # 0..5 as a 1-D tensor
g = torch.randn(2, 3)                    # random normal — model init

# ── 2 · Read the three tags ─────────────────────────
print("x.shape:", x.shape, "dtype:", x.dtype)
print("device:", x.device)

# ── 3 · reshape vs view ─────────────────────────────
xr = x.reshape(3, 2)            # same 6 numbers, new window
xv = x.view(3, 2)              # view needs contiguous memory
print("reshaped:", xr.shape)

# ── 4 · unsqueeze / squeeze ─────────────────────────
v = torch.tensor([2.0, 1.0, 4.0])   # shape (3,)
vu = v.unsqueeze(0)                 # -> shape (1, 3)
vs = vu.squeeze()                   # -> shape (3,)
print("unsqueezed:", vu.shape, "squeezed:", vs.shape)

# ── 5 · index & slice ───────────────────────────────
print("row 1:", x[1, :])            # whole row
print("col 0:", x[:, 0])            # whole column
print("sub-block:", x[0:2, 1:3])    # rows 0-1, cols 1-2

## PT03 · Tensor Math & Broadcasting

In [ ]:
import torch

torch.manual_seed(0)

A = torch.arange(12.).reshape(4, 3)      # (4, 3)
B = torch.arange(12., 24.).reshape(4, 3) # (4, 3)

# --- elementwise: same shape in, same shape out ---
print("A + B:", (A + B).shape)
print("A * B:", (A * B).shape)
print("exp(A):", torch.exp(A).shape)

# --- matmul: inner dims must match, then vanish ---
W = torch.arange(6.).reshape(3, 2)       # (3, 2)
print("A @ W:", (A @ W).shape)           # (4,3)@(3,2) -> (4,2)

# --- reductions: which dimension disappears? ---
print("sum dim=0:", A.sum(dim=0).shape)  # collapses rows -> (3,)
print("sum dim=1:", A.sum(dim=1).shape)  # collapses cols -> (4,)
print("mean dim=1 keepdim:", A.mean(dim=1, keepdim=True).shape)

# --- broadcasting: a (3,) bias onto a (4,3) matrix ---
bias = torch.tensor([100., 200., 300.])  # (3,)
print("after +bias:", (A + bias).shape)  # stretches down 4 rows

# --- the classic shape bug, caught and fixed ---
bad = torch.tensor([1., 2., 3., 4.])     # (4,) — wrong axis
try:
    print((A + bad).shape)
except RuntimeError as e:
    print("RuntimeError:", e)
print("fixed:", (A + bad.reshape(4, 1)).shape)  # (4,1) stretches -> (4,3)

201  302
204  305
207  308
210  311

## PT04 · Autograd — PyTorch Does the Calculus

In [ ]:
import torch

# --- One variable: y = x^2 + 3x ---
x = torch.tensor(2.0, requires_grad=True)
y = x**2 + 3 * x                 # forward: 4 + 6 = 10
y.backward()                     # run the chain rule backward
print(f"y = {y.item()}")
print(f"dy/dx at x=2 -> x.grad = {x.grad.item()} "
      f"(analytic 2x+3 = {2*2 + 3} OK)")

# --- Two variables: both partials at once ---
w = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(1.0, requires_grad=True)
loss = w**2 + b                  # dL/dw = 2w = 4, dL/db = 1
loss.backward()
print(f"grads w,b: {w.grad.item()}, {b.grad.item()}")

# --- Gradients ACCUMULATE by default ---
y2 = x**2 + 3 * x                # rebuild the graph on x
y2.backward()                    # adds on top of the old grad
print(f"after second backward (accumulated): {x.grad.item()}")
x.grad.zero_()                   # clear it in-place
(x**2 + 3 * x).backward()        # fresh backward after zeroing
print(f"after zero_grad: {x.grad.item()}")

# --- Inference: no tape, no graph ---
with torch.no_grad():
    z = x * 5
print(f"no_grad result requires grad? {z.requires_grad}")

## PT05 · Your First Trained Thing — Linear Regression by Hand

In [ ]:
import torch

torch.manual_seed(0)              # reproducible noise

# --- synthetic data: y = 2x + 1 + noise ---
x = torch.linspace(-1.0, 1.0, 100).unsqueeze(1)   # 100 points, shape (100, 1)
true_w, true_b = 2.0, 1.0
noise = 0.1 * torch.randn(x.shape)
y = true_w * x + true_b + noise                   # the messy target

# --- the model: two scalars we will train ---
w = torch.zeros(1, requires_grad=True)
b = torch.zeros(1, requires_grad=True)

# --- one hyperparameter: the step size ---
lr = 0.1

# --- the training loop: 200 steps ---
for step in range(201):
    # 1) predict (forward pass)
    pred = w * x + b

    # 2) measure how wrong we are (MSE)
    loss = ((pred - y) ** 2).mean()

    # 3) blame: fill w.grad and b.grad
    loss.backward()

    # 4) adjust: step downhill, no graph tracking
    with torch.no_grad():
        w -= lr * w.grad
        b -= lr * b.grad
        w.grad.zero_()            # autograd accumulates — reset!
        b.grad.zero_()

    # report progress every 40 steps
    if step % 40 == 0:
        print(f"step {step:>3} | loss {loss.item():.4f} | w {w.item():.2f} b {b.item():.2f}")
print(f"learned: y ≈ {w.item():.2f}x + {b.item():.2f}")

## PT06 · nn.Module & Optimizers — The Grown-Up Loop

In [ ]:
import torch
import torch.nn as nn

# same data as last episode: true line is y = 2x + 1
X = torch.tensor([[1.0], [2.0], [3.0], [4.0]])
y = torch.tensor([[3.0], [5.0], [7.0], [9.0]])

class LinReg(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(1, 1)

    def forward(self, x):
        return self.linear(x)

model = LinReg()
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

for epoch in range(100):
    pred = model(X)
    loss = loss_fn(pred, y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if epoch % 10 == 0:
        print(f"epoch {epoch} loss {loss.item():.4f}")

print(model.state_dict())

## PT07 · A Real Neural Net — MLP on Real Data

In [ ]:
import torch
import torch.nn as nn
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

# two interleaving crescents — no straight line can split them
X, y = make_moons(n_samples=1000, noise=0.2, random_state=0)
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=0)

X_tr = torch.tensor(X_tr, dtype=torch.float32)
X_te = torch.tensor(X_te, dtype=torch.float32)
y_tr = torch.tensor(y_tr, dtype=torch.long)
y_te = torch.tensor(y_te, dtype=torch.long)

model = nn.Sequential(
    nn.Linear(2, 32), nn.ReLU(),
    nn.Linear(32, 32), nn.ReLU(),
    nn.Linear(32, 2),
)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

for epoch in range(201):
    optimizer.zero_grad()
    logits = model(X_tr)
    loss = loss_fn(logits, y_tr)
    loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        preds = logits.argmax(dim=1)
        acc = (preds == y_tr).float().mean()
        print(f"epoch {epoch} loss {loss.item():.3f} acc {acc.item():.2f}")

model.eval()
with torch.no_grad():
    test_logits = model(X_te)
    test_preds = test_logits.argmax(dim=1)
    test_acc = (test_preds == y_te).float().mean()

print(f"final test accuracy: {test_acc.item():.3f}")

## PT08 · Datasets, DataLoaders & GPU

In [ ]:
import time
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader

# --- synthetic data: 2 features, simple linear rule ---
torch.manual_seed(0)
N = 60000
X = torch.randn(N, 2)
y = (X[:, 0] + X[:, 1] > 0).long()

class ToyDataset(Dataset):
    def __init__(self, X, y):
        self.X, self.y = X, y
    def __len__(self):
        return len(self.X)
    def __getitem__(self, i):
        return self.X[i], self.y[i]

dataset = ToyDataset(X, y)
loader = DataLoader(dataset, batch_size=64, shuffle=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")

loss_fn = nn.CrossEntropyLoss()

def make_model():
    return nn.Sequential(
        nn.Linear(2, 2048), nn.ReLU(),
        nn.Linear(2048, 512), nn.ReLU(),
        nn.Linear(512, 2),
    )

def train_epochs(device, epochs=8):
    model = make_model().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    t0 = time.perf_counter()
    for e in range(epochs):
        total = 0.0
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward(); opt.step()
            total += loss.item()
        print(f"  epoch {e+1}/{epochs}  loss {total/len(loader):.3f}")

    return time.perf_counter() - t0

print(f"batches/epoch: {len(loader)}")
print("[cpu]");  cpu_secs  = train_epochs(torch.device("cpu"))
print("[cuda]"); cuda_secs = train_epochs(device)
print(f"[cpu]  {cpu_secs:.1f}s   [cuda] {cuda_secs:.1f}s")

plt.bar(["CPU", "GPU"], [cpu_secs, cuda_secs],
        color=["#888888", "#2ca02c"])
plt.ylabel("seconds"); plt.title("Epoch-loop time: CPU vs GPU")
plt.savefig("speedup.png")

## PT09 · Text Is the Enemy — Tokenization

In [ ]:
import torch

# 1 · Load a public text file (complete works of Shakespeare)
with open("input.txt", "r", encoding="utf-8") as f:
    text = f.read()
print(f"corpus length: {len(text):,} chars")

# 2 · Build a character-level vocabulary
chars = sorted(set(text))
vocab_size = len(chars)
print(f"vocab size: {vocab_size}")

# 3 · Two mirror dictionaries: string <-> integer
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

# 4 · encode / decode are inverse lookups
def encode(s):    return [stoi[c] for c in s]
def decode(ids):  return "".join(itos[i] for i in ids)

# 5 · Round-trip proof on a sample line
print(f"encode('hello') -> {encode('hello')}")
print(f"decode([46, 43, 50, 50, 53]) -> {decode([46, 43, 50, 50, 53])!r}")
sample = "To be, or not to be"
assert decode(encode(sample)) == sample, "round-trip FAILED"
print("round-trip OK ✓")

# 6 · Wrap the whole corpus as a torch.long tensor
data = torch.tensor(encode(text), dtype=torch.long)
print(f"data shape: {tuple(data.shape)}")
print(f"dtype: {data.dtype}")

## PT10 · Embeddings & Loading a Real Dataset

In [ ]:
import torch
import torch.nn as nn
from datasets import load_dataset

# 1 · Load a real corpus from the HuggingFace hub
#     (ships a loader script -> trust_remote_code=True)
ds = load_dataset("karpathy/tiny_shakespeare", trust_remote_code=True)

# 2 · It arrives pre-split; glue the splits into one book
text = "".join(ds[s]["text"][0] for s in ("train", "validation", "test"))
print(f"loaded tiny_shakespeare: {len(text):,} chars")

# 3 · Build the character vocabulary
chars = sorted(set(text))
vocab_size = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for i, c in enumerate(chars)}
print("vocab size:", vocab_size)

# 4 · Encode the whole corpus into one long tensor
encode = lambda s: [stoi[c] for c in s]
data = torch.tensor(encode(text), dtype=torch.long)

# 5 · Train / val split — keep the last 10% for honest eval
n = int(0.9 * len(data))
train_data = data[:n]
val_data   = data[n:]

# 6 · The embedding: a learnable (vocab_size x n_embd) table
n_embd = 32
token_embedding = nn.Embedding(vocab_size, n_embd)

# 7 · Look up a small (B, T) batch -> (B, T, n_embd)
ids = train_data[:4 * 8].view(4, 8)          # (B, T) integers
print("embedded:", token_embedding(ids).shape)
print(f"vector for token 46 ({itos[46]!r}):", token_embedding(torch.tensor(46)).data)

## PT11 · Position & The Training Batch

In [ ]:
# batches.py — turn the encoded corpus into (context, next-token) training batches
import torch
import torch.nn as nn

torch.manual_seed(1337)

# --- data from PT10: the whole corpus encoded as one long tensor of token ids ---
from dataset import data, encode, decode, vocab_size  # data: shape (N,) int64
print(f"corpus tokens: {data.shape[0]}  vocab: {vocab_size}")

# --- hyperparameters that define one batch ---
batch_size = 4     # how many independent windows per step (B)
block_size = 8     # how many tokens of context the model sees (T)
n_embd     = 32    # width of every embedding vector (C)

# --- two learnable lookup tables: WHAT each token is, and WHERE it sits ---
token_embedding_table = nn.Embedding(vocab_size, n_embd)
position_embedding_table = nn.Embedding(block_size, n_embd)

def get_batch():
    # pick batch_size random start offsets that leave room for a full block
    ix = torch.randint(len(data) - block_size, (batch_size,))
    # x = tokens[i:i+block]   y = same block shifted one step right
    x = torch.stack([data[i      : i + block_size]     for i in ix])
    y = torch.stack([data[i + 1  : i + block_size + 1] for i in ix])
    return x, y

# --- run it once ---
x, y = get_batch()
print("x shape:", x.shape, " y shape:", y.shape)

# --- look up token vectors and position vectors, then add them ---
tok_emb = token_embedding_table(x)                       # (B, T, C)
pos_emb = position_embedding_table(torch.arange(block_size))  # (T, C)
combined = tok_emb + pos_emb                             # broadcast -> (B, T, C)
print("combined emb shape:", combined.shape)

# --- decode one window so the (context -> target) pairs are readable ---
for t in (7, 8):                       # grow the context: 7 chars, then 8
    ctx, tgt = x[0, :t], y[0, t - 1]   # tgt = token right after ctx (= x[0, t])
    print(f"context {decode(ctx.tolist())!r} -> target {decode([tgt.item()])!r}")

## PT12 · Self-Attention From Scratch — Q, K, V

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(1337)
B, T, C = 4, 8, 32          # batch, time (tokens), channels

class Head(nn.Module):
    """One self-attention head."""
    def __init__(self, head_size):
        super().__init__()
        self.key   = nn.Linear(C, head_size, bias=False)
        self.query = nn.Linear(C, head_size, bias=False)
        self.value = nn.Linear(C, head_size, bias=False)

    def forward(self, x):
        k = self.key(x)                              # (B, T, head_size)
        q = self.query(x)                            # (B, T, head_size)
        v = self.value(x)                            # (B, T, head_size)
        scores = q @ k.transpose(-2, -1) * k.shape[-1] ** -0.5   # (B, T, T)
        weights = F.softmax(scores, dim=-1)          # (B, T, T)
        out = weights @ v                            # (B, T, head_size)
        return out, weights

x = torch.randn(B, T, C)
head = Head(head_size=16)

with torch.no_grad():
    out, weights = head(x)

row = weights[0, 4]                          # token 4 attends over all 8 tokens
print("attn weights (row 4, sums to 1):",
      [round(w, 2) for w in row.tolist()])
print("row 4 sum:", round(row.sum().item(), 4))
print("output shape:", out.shape)
top = torch.topk(row, 2).indices.tolist()
print(f"token 4 mostly attends to tokens {top[0]} and {top[1]}")

## PT13 · Causal Masking — No Peeking at the Future

In [ ]:
import torch
import torch.nn.functional as F

torch.manual_seed(13)

# --- setup: 8 tokens, tiny head dimension ---
T, d_k = 8, 4

# fake attention scores (in a real model: Q @ K.T / sqrt(d_k))
scores = torch.randn(T, T) / (d_k ** 0.5)

# --- build the causal mask ---
mask = torch.tril(torch.ones(T, T))      # 1 on/below diagonal, 0 above

# poison the future: upper triangle -> -inf before softmax
scores = scores.masked_fill(mask == 0, float('-inf'))

# --- softmax along each row turns scores into weights ---
weights = F.softmax(scores, dim=-1)

# --- show the result ---
torch.set_printoptions(precision=2, sci_mode=False)
print(weights)

assert torch.all(weights.triu(diagonal=1) == 0), "future leaked!"
print("✓ no future leakage")

## PT14 · Multi-Head Attention

In [ ]:
# multihead.py — multi-head causal self-attention
import torch
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(1337)
batch, block_size, n_embd = 4, 8, 32   # B, T, C
n_head = 4
head_size = n_embd // n_head            # 32 // 4 = 8

class Head(nn.Module):
    """One causal self-attention head (from PT13)."""
    def __init__(self, head_size):
        super().__init__()
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)            # (B, T, head_size)
        q = self.query(x)          # (B, T, head_size)
        wei = q @ k.transpose(-2, -1) * head_size ** -0.5   # (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        wei = F.softmax(wei, dim=-1)
        v = self.value(x)          # (B, T, head_size)
        return wei @ v             # (B, T, head_size)

class MultiHeadAttention(nn.Module):
    """Several causal heads in parallel, concatenated and projected."""
    def __init__(self, n_head, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(n_head)])
        self.proj  = nn.Linear(head_size * n_head, n_embd)   # 8*4 -> 32

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)  # (B, T, 32)
        return self.proj(out)                                # (B, T, 32)

# --- run one batch and watch the shapes ---
x = torch.randn(batch, block_size, n_embd)   # (4, 8, 32)
mha = MultiHeadAttention(n_head, head_size)

one_head_out = mha.heads[0](x)               # peek at a single head
concatenated = torch.cat([h(x) for h in mha.heads], dim=-1)  # (4, 8, 32)
combined     = mha(x)                         # concat + project

print(f"num heads: {n_head} | head size: {head_size}")
print("per-head output:", one_head_out.shape)
print("concatenated:", concatenated.shape)
print("after projection:", combined.shape)
print(f"{n_head} heads, {n_head} views of the same sentence")

## PT15 · The Transformer Block — Attention + MLP + Residuals

In [ ]:
import torch
import torch.nn as nn


class MultiHeadAttention(nn.Module):
    """From PT14 — mixes tokens; shape in == shape out."""
    def __init__(self, n_embd, n_head):
        super().__init__()
        self.attn = nn.MultiheadAttention(n_embd, n_head, batch_first=True)

    def forward(self, x):
        out, _ = self.attn(x, x, x)   # self-attention: Q = K = V = x
        return out                     # (B, T, n_embd)


class FeedForward(nn.Module):
    """Per-token MLP: Linear -> ReLU -> Linear, 4x expansion."""
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
        )
    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """One transformer block: communicate, then compute."""
    def __init__(self, n_embd, n_head):
        super().__init__()
        self.attn = MultiHeadAttention(n_embd, n_head)
        self.ffwd = FeedForward(n_embd)
        self.ln1  = nn.LayerNorm(n_embd)
        self.ln2  = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))   # talk: pre-norm + residual
        x = x + self.ffwd(self.ln2(x))   # think: pre-norm + residual
        return x


if __name__ == "__main__":
    torch.manual_seed(0)
    x = torch.randn(4, 8, 32)            # (batch, tokens, channels)
    block = Block(n_embd=32, n_head=4)
    out = block(x)

    print("block input: ", x.shape)
    print("block output:", out.shape)
    print("shape preserved -> blocks are stackable")
    n = sum(p.numel() for p in block.parameters())
    print(f"params in one block: {n:,}")

## PT16 · Assembling Our Tiny GPT

In [ ]:
# model.py — assemble token+pos embeddings, a stack of Blocks,
# a final norm, and a vocab-size head into one GPT class.
import torch
import torch.nn as nn
from torch.nn import functional as F

from block import Block            # the Transformer block from PT15

# --- config: the five knobs that size the model ---
vocab_size = 65                    # 65 unique characters in our toy text
block_size = 8                     # max positions the model can attend over
n_embd     = 32                    # width of every token / position vector
n_layer    = 4                     # how many Blocks we stack
n_head     = 4                     # attention heads inside each Block


class GPT(nn.Module):
    def __init__(self):
        super().__init__()
        # two lookup tables: one for tokens, one for positions
        self.token_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb   = nn.Embedding(block_size, n_embd)

        # N identical Blocks, chained into one tower
        self.blocks = nn.Sequential(
            *[Block(n_embd, n_head) for _ in range(n_layer)]
        )

        # final normalization, then project to one score per token
        self.ln_f    = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok = self.token_emb(idx)                       # (B, T, n_embd)
        pos = self.pos_emb(torch.arange(T, device=idx.device))  # (T, n_embd)
        x = tok + pos                                   # broadcast over batch

        x = self.blocks(x)                              # run the whole tower
        x = self.ln_f(x)                                # final LayerNorm
        logits = self.lm_head(x)                        # (B, T, vocab_size)

        if targets is None:
            return logits, None

        # flatten so cross-entropy can compare position-by-position
        B, T, V = logits.shape
        loss = F.cross_entropy(logits.view(B * T, V), targets.view(B * T))
        return logits, loss

if __name__ == "__main__":
    model = GPT()
    n_params = sum(p.numel() for p in model.parameters())
    print(f"model: GPT | blocks: {n_layer} | heads: {n_head} | n_embd: {n_embd}")
    print(f"total parameters: {n_params / 1e6:.2f} M")

    idx     = torch.randint(0, vocab_size, (4, block_size))   # fake batch
    targets = torch.randint(0, vocab_size, (4, block_size))   # fake answers
    logits, loss = model(idx, targets)
    print(f"logits shape: {logits.shape}")
    print(f"loss (untrained): {loss.item():.2f} (random ≈ ln(65)=4.17)")

## PT17 · Training the Tiny GPT — Loss Falls, Words Appear

In [ ]:
import torch
from model import TinyGPT
from data import get_batch, decode

# --- hyperparameters ---
max_iters     = 5000
eval_interval = 500
eval_iters    = 200
learning_rate = 3e-4
device = "cuda" if torch.cuda.is_available() else "cpu"

# build the model and move it to the device
model = TinyGPT().to(device)

# AdamW: adaptive, per-parameter steps + proper weight decay
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ("train", "val"):
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):

            xb, yb = get_batch(split)
            _, loss = model(xb, yb)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

# --- training loop: forward, loss, backward, step ---
for step in range(max_iters):
    # periodically report a stable loss + a live sample
    if step % eval_interval == 0:
        losses = estimate_loss()
        ctx = torch.zeros((1, 1), dtype=torch.long, device=device)
        sample = decode(model.generate(ctx, max_new_tokens=40)[0].tolist())
        print(f"step {step:4d}  loss {losses['train']:.2f}  -> {sample!r}")

    xb, yb = get_batch("train")     # 1 · grab a batch
    _, loss = model(xb, yb)         # 2 · forward + loss
    optimizer.zero_grad()           # 3 · clear old grads
    loss.backward()                 #     backward
    optimizer.step()                #     apply the step

torch.save(model.state_dict(), "gpt_tiny.pt")
print("saved -> gpt_tiny.pt")

## PT18 · Generation & Sampling — Watch It Talk

In [ ]:
import torch
import torch.nn.functional as F
from model import GPT, encode, decode, block_size


@torch.no_grad()
def generate(model, idx, max_new_tokens, temperature=1.0, top_k=None):
    model.eval()
    for _ in range(max_new_tokens):
        # crop context to the last block_size tokens
        idx_cond = idx[:, -block_size:]
        # forward the model to get logits for every position
        logits, _ = model(idx_cond)
        # keep only the final step, scaled by temperature
        logits = logits[:, -1, :] / temperature
        # optional top-k filter: -inf everything below the k-th best
        if top_k is not None:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = -float("inf")
        probs = F.softmax(logits, dim=-1)
        # sample one token from the distribution
        idx_next = torch.multinomial(probs, num_samples=1)
        idx = torch.cat((idx, idx_next), dim=1)
    return idx


# ---- load checkpoint & stream ----
device = "cuda" if torch.cuda.is_available() else "cpu"
ckpt = torch.load("tiny_gpt.pt", map_location=device)
model = GPT(ckpt["config"]).to(device)
model.load_state_dict(ckpt["model"])

prompt = "ROMEO:"
ids = torch.tensor([encode(prompt)], dtype=torch.long, device=device)

print(f"prompt: {prompt}")
print("--- temp 0.8, top-k 40 ---")
out = generate(model, ids, max_new_tokens=200, temperature=0.8, top_k=40)
print(decode(out[0].tolist()))

print("--- temp 1.4 (wilder) ---")
out = generate(model, ids, max_new_tokens=200, temperature=1.4, top_k=40)
print(decode(out[0].tolist()))

## PT19 · From Tiny GPT to ChatGPT — What You Built & What's Next

In [ ]:
# scale_up.py — the SAME model.py, scaled to frontier sizes.
# We import our GPT unchanged; only the CONFIG numbers grow.
from model import GPT          # the exact class from PT15–PT17

# --- Config dicts: identical architecture, different integers ---
TINY = dict(n_layer=4,  n_head=4,  n_embd=48,    block_size=8,    vocab_size=65)
GPT2 = dict(n_layer=12, n_head=12, n_embd=768,   block_size=1024, vocab_size=50257)
GPT3 = dict(n_layer=96, n_head=96, n_embd=12288, block_size=2048, vocab_size=50257)

def count_params(cfg):
    """Total learnable weights for a given config (GPT-style, tied embeddings)."""
    L, E = cfg["n_layer"], cfg["n_embd"]
    V, T = cfg["vocab_size"], cfg["block_size"]
    emb   = V * E + T * E                 # token + position embeddings
    block = 12 * E * E                    # attn + MLP weights per layer
    return emb + L * block                # unembedding ties to token emb -> counted once

def fmt(n):
    for unit in ["", "K", "M", "B"]:
        if abs(n) < 1000: return f"{n:.2f}{unit}".rstrip("0").rstrip(".")
        n /= 1000
    return f"{n:.2f}T"

def print_comparison():
    rows = [("layers", "n_layer"), ("heads", "n_head"),
            ("n_embd", "n_embd"), ("context", "block_size")]

        names = {"TINY": TINY, "GPT2": GPT2, "GPT3": GPT3}
        print(f"{'':10}" + "".join(f"{k:>12}" for k in names))
        for label, key in rows:
            print(f"{label:10}" + "".join(f"{cfg[key]:>12,}" for cfg in names.values()))
        print(f"{'params':10}" + "".join(f"{fmt(count_params(c)):>12}" for c in names.values()))
        print("\nSame architecture. ~1000x bigger to reach GPT-2. That's the whole secret.")

# --- STUB 1: real subword tokenizer (BPE) — the next upgrade ---
# import tiktoken                                  # OpenAI's BPE
# enc = tiktoken.get_encoding("gpt2")             # 50257-token vocab
# ids = enc.encode("scaling is the whole secret") # -> [259, 1416, ...]
# Same interface our model expects — just smarter token ids.

# --- STUB 2: fine-tuning loop — turn a base model into an assistant ---
# import torch
# model = GPT(**GPT2); model.load_state_dict(torch.load("pretrained.pt"))
# for prompt, answer in instruction_pairs:        # not raw text — pairs
#     logits, loss = model(encode(prompt), targets=encode(answer))
#     loss.backward(); opt.step(); opt.zero_grad()
# (RLHF — ranking answers with human feedback — comes after this.)

if __name__ == "__main__":
    print_comparison()
    tiny = GPT(**TINY)                            # the one we actually trained
    print(f"\nours: {fmt(count_params(TINY))} params, ctx {TINY['block_size']}")
    print("the same file you wrote — just turn the numbers up.")
    # next: point CONFIG at a real dataset and scale.

## ▶ Putting it together — a runnable tiny char-GPT
Self-contained: downloads tiny-shakespeare, trains a small GPT for a few hundred steps, and samples text. Runs on Colab (GPU recommended: Runtime → Change runtime type → GPU).

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F, urllib.request, os
torch.manual_seed(1337)
dev = "cuda" if torch.cuda.is_available() else "cpu"

# ---- data ----
if not os.path.exists("input.txt"):
    urllib.request.urlretrieve("https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt", "input.txt")
text = open("input.txt").read()
chars = sorted(set(text)); V = len(chars)
stoi = {c:i for i,c in enumerate(chars)}; itos = {i:c for c,i in stoi.items()}
enc = lambda s: [stoi[c] for c in s]; dec = lambda l: "".join(itos[i] for i in l)
data = torch.tensor(enc(text), dtype=torch.long)
n = int(0.9*len(data)); tr, va = data[:n], data[n:]

B, T, n_embd, n_head, n_layer = 32, 64, 128, 4, 4
def get_batch(split):
    d = tr if split=="train" else va
    ix = torch.randint(len(d)-T, (B,))
    x = torch.stack([d[i:i+T] for i in ix]); y = torch.stack([d[i+1:i+T+1] for i in ix])
    return x.to(dev), y.to(dev)

class Block(nn.Module):
    def __init__(s):
        super().__init__()
        s.attn = nn.MultiheadAttention(n_embd, n_head, batch_first=True)
        s.ff = nn.Sequential(nn.Linear(n_embd,4*n_embd), nn.ReLU(), nn.Linear(4*n_embd,n_embd))
        s.l1, s.l2 = nn.LayerNorm(n_embd), nn.LayerNorm(n_embd)
        s.register_buffer("mask", torch.triu(torch.ones(T,T), 1).bool())
    def forward(s, x):
        a,_ = s.attn(s.l1(x), s.l1(x), s.l1(x), attn_mask=s.mask[:x.size(1),:x.size(1)])
        x = x + a; x = x + s.ff(s.l2(x)); return x

class GPT(nn.Module):
    def __init__(s):
        super().__init__()
        s.tok = nn.Embedding(V, n_embd); s.pos = nn.Embedding(T, n_embd)
        s.blocks = nn.Sequential(*[Block() for _ in range(n_layer)])
        s.lnf = nn.LayerNorm(n_embd); s.head = nn.Linear(n_embd, V)
    def forward(s, idx, targets=None):
        Tn = idx.size(1)
        x = s.tok(idx) + s.pos(torch.arange(Tn, device=idx.device))
        x = s.head(s.lnf(s.blocks(x)))
        loss = None if targets is None else F.cross_entropy(x.view(-1,V), targets.view(-1))
        return x, loss
    @torch.no_grad()
    def generate(s, idx, n_new, temp=0.8, top_k=40):
        s.eval()
        for _ in range(n_new):
            logits,_ = s(idx[:, -T:]); logits = logits[:, -1, :]/temp
            v,_ = torch.topk(logits, top_k); logits[logits < v[:, [-1]]] = -float("inf")
            p = F.softmax(logits, -1); idx = torch.cat([idx, torch.multinomial(p,1)], 1)
        return idx

model = GPT().to(dev)
print("params:", sum(p.numel() for p in model.parameters())/1e6, "M")
opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
for step in range(500):
    x,y = get_batch("train"); _,loss = model(x,y)
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 100 == 0: print(f"step {step}  loss {loss.item():.3f}")

ctx = torch.zeros((1,1), dtype=torch.long, device=dev)
print(dec(model.generate(ctx, 300)[0].tolist()))